# LLM 6일차 (1) — LangGraph 입문 : 상태(State) · 노드 · 엣지 · TypedDict 로 상태 정의하기

수업 필기 `LangGraph_실습1`(그래프를 만들다 멈춤) · `LangGraph_실습1_1`(실행까지 완료)을 정리 · 개선한 노트입니다.
(교재 `AI Agent` **p.72~87 · 92~93** — LangChain 의 한계 · LangGraph 정의 · 노드 · 엣지 · 상태 · 타입 힌트 · TypedDict · 노드의 상태 공유 · 실습)

```python
class AgentState(TypedDict):                                  # ① 상태 = 모든 노드가 같이 쓰는 화이트보드의 양식
    messages: Annotated[list[BaseMessage], add_messages]      #    대화는 '덧붙이기'
    tool_output: str | None

graph = StateGraph(AgentState)                                # ② 설계도
graph.add_node('call_llm_node', call_llm)                     #    노드 = 함수 (상태를 받아 '바꿀 부분' 을 dict 로 돌려줌)
graph.add_edge(START, 'call_llm_node'); graph.add_edge('call_llm_node', 'call_tool_node'); …
agent = graph.compile()                                       # ③ 실행 가능한 그래프
agent.invoke({'messages': [HumanMessage('nvidia 2026년도 2분기 매출액은?')]})
```

- Pydantic `BaseModel` 로 상태를 정의하는 실습은 [`langgraph_pydantic_state.ipynb`](./langgraph_pydantic_state.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 오늘 실습은 LLM 을 부르지 않고 **노드 함수가 정해 둔 메시지를 돌려주는 모의 실행**이라 **API 키 없이 전부 실제로 실행**했다. `# 📼` 셀만 수업 시간 출력을 옮긴 것이다(메시지 `id` 는 실행마다 달라진다).

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langgraph` | `StateGraph` · `START` · `END` · `add_messages` | `pip install langgraph` |
| `langchain-core` | `HumanMessage` · `AIMessage` · `ToolMessage` | langgraph 와 함께 설치됨 |

**API 키 · 인터넷 필요 없음** (정리 환경 : `langgraph 1.2.21` · `langchain-core 1.8.0`). 맨 위부터 **모두 실행**. 그래프 그림은 Mermaid 글자로 출력한다(`draw_mermaid_png()` 는 인터넷이 필요).

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph

In [2]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import add_messages, StateGraph, START, END

## 1. 왜 LangGraph 인가 (교재 p.73~76)

| | LangChain 체인 (LCEL) | LangGraph |
|---|---|---|
| 모양 | `prompt \| llm \| parser` — 한 방향 컨베이어 벨트 | **노드**(일하는 함수) + **엣지**(다음에 갈 곳) |
| 갈림길 · 되돌아가기 | 어렵다 | 조건부 엣지(`add_conditional_edges`)로 분기 · 반복 |
| 중간 기록 | 체인 밖에서 따로 관리 | **상태(State)** — 모든 노드가 읽고 쓰는 공유 메모리 |

> **📝 교재 p.73 보충** 'LangChain 은 단방향 비순환 · 기억 능력이 없다' 는 **LCEL 체인** 이야기다. Day 047 에서 쓴 LangChain 1.0 의 `create_agent` 는 안에서 이미 LangGraph 그래프(`model ⇄ tools` 반복)로 동작했고(`get_graph().draw_mermaid()` 로 확인), Day 047 챗봇처럼 `MessagesPlaceholder` 로 대화 기록도 넘길 수 있다. LangGraph 를 배우는 이유는 그 반복 · 분기 · 기록을 **직접 설계**하기 위해서다.

## 2. 타입 힌트는 '안내문' 이다 (교재 p.83~84)

In [3]:
def agent_node(retries: int, user_name: str | None = None) -> str:
    return f'retries={retries!r} ({type(retries).__name__}), user_name={user_name!r}'

print(agent_node(3, '김철수'))
print(agent_node('세 번', 123))          # 힌트와 다른 값을 넣어도 파이썬은 그냥 실행한다
print(agent_node.__annotations__)       # 힌트는 함수에 '기록' 만 된다

retries=3 (int), user_name='김철수'
retries='세 번' (str), user_name=123
{'retries': <class 'int'>, 'user_name': str | None, 'return': <class 'str'>}


- 타입 힌트는 사람과 편집기(VS Code · PyCharm · mypy)에게 주는 **안내**다. 실행 중에 막아 주지는 않는다 — 막는 것은 Pydantic(다음 노트)

## 3. `TypedDict` — 항목이 정해진 dict (교재 p.85 · 필기 연습)

In [4]:
class User(TypedDict):
    id: int
    name: str
    email: str
    is_active: bool

dict_data3: User = {'id': 1, 'name': '김철수', 'email': 'chulsoo@example.com', 'is_active': True}
dict_data4 = User(id=2, name='이영희', email='yh@example.com', is_active=False)
print(dict_data4, type(dict_data4))                   # 결과는 그냥 dict

wrong = User(id='삼', name=None, email=3)            # 자료형이 틀리고 is_active 가 빠져도
print(wrong, type(wrong))                             # 오류 없이 만들어진다
print('정해 둔 항목 :', User.__annotations__)

{'id': 2, 'name': '이영희', 'email': 'yh@example.com', 'is_active': False} <class 'dict'>
{'id': '삼', 'name': None, 'email': 3} <class 'dict'>
정해 둔 항목 : {'id': <class 'int'>, 'name': <class 'str'>, 'email': <class 'str'>, 'is_active': <class 'bool'>}


- `TypedDict` 는 '이런 키와 자료형을 쓰자' 는 **설문지 양식**이다. 실행 결과는 평범한 `dict` 이고, 틀린 값 · 빠진 키를 막지 않는다(교재 p.89 표 '예방(X)' 그대로)

## 4. 상태와 리듀서 — `Annotated[list, add_messages]` (교재 p.86)

In [5]:
class PlainState(TypedDict):                  # 리듀서 없음
    messages: list[BaseMessage]

class AgentState(TypedDict):                  # 리듀서 add_messages
    messages: Annotated[list[BaseMessage], add_messages]
    tool_output: str | None

def reply(state):
    return {'messages': [AIMessage(content='반갑습니다! 무엇을 도와드릴까요?')]}

for State in [PlainState, AgentState]:
    g = StateGraph(State)
    g.add_node('reply', reply); g.add_edge(START, 'reply'); g.add_edge('reply', END)
    out = g.compile().invoke({'messages': [HumanMessage(content='안녕')]})
    print(f'{State.__name__:10s} →', [f'{type(m).__name__}({m.content})' for m in out['messages']])

PlainState → ['AIMessage(반갑습니다! 무엇을 도와드릴까요?)']
AgentState → ['HumanMessage(안녕)', 'AIMessage(반갑습니다! 무엇을 도와드릴까요?)']


In [6]:
# add_messages 가 하는 일 — 직접 불러 보기
old = [HumanMessage(content='안녕', id='h1')]
print('덧붙이기     :', [m.content for m in add_messages(old, [AIMessage(content='반가워요', id='a1')])])
print('같은 id 교체 :', [m.content for m in add_messages(old + [AIMessage(content='반가워요', id='a1')], [AIMessage(content='반갑습니다!', id='a1')])])
print('글자도 받음  :', add_messages([], '안녕하세요')[0].__class__.__name__)

덧붙이기     : ['안녕', '반가워요']
같은 id 교체 : ['안녕', '반갑습니다!']
글자도 받음  : HumanMessage


- 리듀서가 없으면 노드가 돌려준 값이 **덮어쓴다** — 필기 설명 '`list[BaseMessage]` 만 쓰면 기존 대화가 새 메시지로 대체' 가 위 결과 그대로다
- `add_messages` 는 ① 뒤에 덧붙이고 ② **같은 `id` 면 교체**하고 ③ 글자 · dict 도 메시지로 바꿔 준다
- `Annotated[자료형, 부가 정보]` 는 타입 힌트에 정보를 덧붙이는 문법이고, LangGraph 는 그 부가 정보 자리의 함수를 **리듀서**(값을 합치는 방법)로 쓴다. 리듀서가 없는 `tool_output` 은 마지막 값으로 덮어쓴다

## 5. 노드 함수 (교재 p.87 · 필기)

In [7]:
# 📝 교재 p.87 · p.91 의 코드 그대로 — ToolMessage 에 tool_call_id 가 없다
try:
    ToolMessage(content='도구 실행 결과: 데이터 처리 완료.')
except Exception as e:
    print('교재 코드 :', type(e).__name__, e)

# 📝 필기 LangGraph_실습1 의 call_llm 그대로 — tool_calls 에 id 가 없다
try:
    AIMessage(content='', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'nvidia 2026년도 2분기 매출액'}}])
except Exception as e:
    print('필기 코드 :', type(e).__name__, e)

교재 코드 : KeyError 'tool_call_id'
필기 코드 : TypeError tool_call() missing 1 required keyword-only argument: 'id'


- `ToolMessage` 는 '어느 도구 호출에 대한 답인지' 를 **`tool_call_id`** 로 꼭 밝혀야 하고, `AIMessage` 의 `tool_calls` 에는 **`id`** 가 꼭 있어야 한다 — 둘이 같은 값이어야 짝이 맞는다(Day 047 에이전트 기록의 `call_…` id)
- 필기 `LangGraph_실습1` 은 그래프를 만들고 멈춰서 이 오류가 드러나지 않았다 — 실행하면 `call_llm_node` 에서 `TypeError` 로 멈춘다

In [8]:
CALL_ID = 'call_search_001'                               # AIMessage 의 tool_calls id 와 ToolMessage 의 tool_call_id 를 같게

def call_llm(state: AgentState) -> dict:
    print('---LLM 호출 노드 실행---')
    messages = state['messages']
    last_user_message = messages[-1].content if messages and isinstance(messages[-1], HumanMessage) else ''
    print(f'사용자 입력: {last_user_message}')
    # 실제로는 LLM 이 판단 — 여기서는 '검색 도구를 쓰겠다' 는 응답을 직접 만든다
    return {'messages': [AIMessage(content='', tool_calls=[
        {'name': 'tavily_search', 'args': {'query': last_user_message}, 'id': CALL_ID}])]}

def call_tool(state: AgentState) -> dict:
    print('---도구 노드 실행---')
    call = state['messages'][-1].tool_calls[0]           # 직전 AIMessage 가 요청한 도구 호출
    tool_result = f"'{call['args']['query']}' 검색 완료 (모의 결과)"
    return {'messages': [ToolMessage(content=tool_result, tool_call_id=call['id'])], 'tool_output': tool_result}

def summarize_result(state: AgentState) -> dict:
    print('---결과 요약 노드 실행---')
    summary = f"도구 실행 결과({state['tool_output']})를 요약했습니다."
    return {'messages': [AIMessage(content=summary)]}

- 노드는 상태 **전체**를 받아서, 바꿀 **항목만** dict 로 돌려준다. LangGraph 가 그 dict 를 리듀서 규칙대로 상태에 반영한다(`messages` 는 덧붙이고, `tool_output` 은 덮어쓰기)
- 수업 코드는 `tool_call_id='searcg_tool_call_id_123'`(오타 · AIMessage 와 짝이 안 맞는 id) — 위처럼 직전 `AIMessage.tool_calls` 의 `id` 를 꺼내 쓴다

## 6. 그래프 만들기 · 컴파일 (교재 p.92)

In [9]:
graph = StateGraph(AgentState)                    # 설계도 (아직 실행 못 함)
graph.add_node('call_llm_node', call_llm)
graph.add_node('call_tool_node', call_tool)
graph.add_node('summarize_node', summarize_result)
graph.add_edge(START, 'call_llm_node')            # = graph.set_entry_point('call_llm_node') (수업 방식)
graph.add_edge('call_llm_node', 'call_tool_node')
graph.add_edge('call_tool_node', 'summarize_node')
graph.add_edge('summarize_node', END)
agent = graph.compile()                           # 실행 가능한 그래프 (Runnable — invoke · stream)
print(type(graph).__name__, '→', type(agent).__name__)
print(agent.get_graph().draw_mermaid())

StateGraph → CompiledStateGraph
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	call_llm_node(call_llm_node)
	call_tool_node(call_tool_node)
	summarize_node(summarize_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> call_llm_node;
	call_llm_node --> call_tool_node;
	call_tool_node --> summarize_node;
	summarize_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



- 필기 출력의 `<langgraph.graph.state.StateGraph at 0x…>` 는 셀의 마지막 줄 `graph.add_edge(...)` 가 **그래프 자신을 돌려줘서** 찍힌 것이다(메서드 체이닝용). 오류가 아니다
- `set_entry_point(이름)` 은 `add_edge(START, 이름)` 과 같다. 요즘 공식 문서는 `START` 를 쓴다

## 7. 실행 — `invoke` 와 `stream`

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

print("=== TypedDict 기반 에이전트 실행 ===")
user_question = "nvidia 2026년도 2분기 매출액은?"
initial_state = {"messages": [HumanMessage(content=user_question)]}
final_state = agent.invoke(initial_state)
print(f"\n최종 상태: \n{final_state}")
print('-'*80)
print(f"최종 상태 메시지: {final_state['messages']}")
print('-'*80)
print(f"최종 LLM 출력: {final_state['messages'][-1].content}")

=== TypedDict 기반 에이전트 실행 ===
---LLM 호출 노드 실행---
--------------------------------------------------------------------------------
사용자 입력: nvidia 2026년도 2분기 매출액은?
--------------------------------------------------------------------------------
---도구 노드 실행---
--------------------------------------------------------------------------------
도구 실행 중...
---결과 요약 노드 실행---
--------------------------------------------------------------------------------

최종 상태: 
{'messages': [HumanMessage(content='nvidia 2026년도 2분기 매출액은?', additional_kwargs={}, response_metadata={}, id='81ccb142-793c-4171-a8bc-348f33b59688'), AIMessage(content='도구 사용을 요청합니다.', additional_kwargs={}, response_metadata={}, id='1bf996c8-918d-4121-ba3c-d2e7fa4d3017', tool_calls=[], invalid_tool_calls=[]), ToolMessage(content='도구 실행 완료.', id='5026ca86-892d-490a-94a5-273491582c68', tool_call_id='searcg_tool_call_id_123'), AIMessage(content="도구 실행 결과 '도구 실행 완료.'를 요약했습니다.", additional_kwargs={}, response_metadata={}, id='f3c30854-d7aa-4b

In [11]:
final_state = agent.invoke({'messages': [HumanMessage(content='nvidia 2026년도 2분기 매출액은?')]})
print()
for m in final_state['messages']:
    extra = f" tool_calls={[(c['name'], c['id']) for c in m.tool_calls]}" if getattr(m, 'tool_calls', None) else ''
    extra += f" tool_call_id={m.tool_call_id}" if isinstance(m, ToolMessage) else ''
    print(f'{type(m).__name__:12s} {m.content!r}{extra}')
print('tool_output :', final_state['tool_output'])

---LLM 호출 노드 실행---
사용자 입력: nvidia 2026년도 2분기 매출액은?
---도구 노드 실행---
---결과 요약 노드 실행---

HumanMessage 'nvidia 2026년도 2분기 매출액은?'
AIMessage    '' tool_calls=[('tavily_search', 'call_search_001')]
ToolMessage  "'nvidia 2026년도 2분기 매출액은?' 검색 완료 (모의 결과)" tool_call_id=call_search_001
AIMessage    "도구 실행 결과('nvidia 2026년도 2분기 매출액은?' 검색 완료 (모의 결과))를 요약했습니다."
tool_output : 'nvidia 2026년도 2분기 매출액은?' 검색 완료 (모의 결과)


In [12]:
# stream(stream_mode='updates') — 노드마다 '상태에 무엇을 바꿨는지' 를 하나씩
for step in agent.stream({'messages': [HumanMessage(content='nvidia 2026년도 2분기 매출액은?')]}, stream_mode='updates'):
    for node, update in step.items():
        print(f"[{node}] →", {k: ([type(m).__name__ for m in v] if k == 'messages' else v) for k, v in update.items()})

---LLM 호출 노드 실행---
사용자 입력: nvidia 2026년도 2분기 매출액은?
[call_llm_node] → {'messages': ['AIMessage']}
---도구 노드 실행---
[call_tool_node] → {'messages': ['ToolMessage'], 'tool_output': "'nvidia 2026년도 2분기 매출액은?' 검색 완료 (모의 결과)"}
---결과 요약 노드 실행---
[summarize_node] → {'messages': ['AIMessage']}


- `invoke` 는 마지막 상태만, `stream` 은 **노드가 끝날 때마다** 바뀐 부분을 보여 준다 — 그래프가 어떻게 흘렀는지 확인할 때 쓴다
- 최종 `messages` = Human → AI(tool_calls) → Tool → AI. 수업 실행본의 첫 AIMessage 는 `tool_calls=[]` 인 평범한 글('도구 사용을 요청합니다.')이었다. 모의 실행이라 돌아가지만, 실제 OpenAI API 에 이 순서를 보내면 '`tool` 메시지는 `tool_calls` 를 가진 메시지 뒤에만 올 수 있다' 는 오류가 난다 → 위처럼 `tool_calls` 와 id 를 맞춘다

## 8. 📝 TypedDict 상태는 빠진 키를 채워 주지 않는다

In [13]:
g = StateGraph(AgentState)
g.add_node('summarize_node', summarize_result)    # 도구 노드를 건너뛰고 바로 요약하면
g.add_edge(START, 'summarize_node'); g.add_edge('summarize_node', END)
try:
    g.compile().invoke({'messages': [HumanMessage(content='안녕')]})
except KeyError as e:
    print('KeyError :', e, "← tool_output 을 아무 노드도 쓰지 않아 상태에 없다")

---결과 요약 노드 실행---
KeyError : 'tool_output' ← tool_output 을 아무 노드도 쓰지 않아 상태에 없다


- `tool_output: str | None` 이라고 적어도 **기본값 None 이 생기지 않는다**(TypedDict 는 기본값이 없다). 순서가 바뀌는 그래프라면 `state.get('tool_output')` 으로 읽거나, 기본값을 주는 Pydantic 상태를 쓴다(다음 노트)

> **📝 고친 부분 (필기 `LangGraph_실습1` · `LangGraph_실습1_1` · 교재)**
> - `LangGraph_실습1` : `AIMessage(tool_calls=[{'name': …, 'args': …}])` 에 `id` 가 없어 **실행하면 `TypeError`** — 컴파일 · 실행 전에 멈춰 있어 드러나지 않았다 → `id` 추가
> - 두 필기 : `ToolMessage(tool_call_id='searcg_tool_call_id_123')` — 오타(search) · 앞의 `AIMessage` 에 그 id 의 도구 호출이 없음 → 직전 `tool_calls[0]['id']` 를 사용
> - `LangGraph_실습1_1` : 도구 호출이 없는 `AIMessage` 다음에 `ToolMessage` — 실제 LLM API 에는 보낼 수 없는 순서
> - `LangGraph_실습1_1` : 노드 설명 `{'messages':[BaseMessage], 'tool_output']}` — 괄호 짝이 안 맞는 표기 → `{'messages': [...], 'tool_output': ...}`
> - 교재 p.87 · p.91 `ToolMessage(content=tool_result)` → `tool_call_id` 가 없어 **`KeyError: 'tool_call_id'`**
> - 교재 p.86 주석 'TypedDict 클래스 **함수** 상속' → 클래스 상속
> - 교재 p.74 제목 `LangGrpah` → LangGraph · 머리글 `Famiy` → Family

## 정리

| 할 일 | 코드 |
|---|---|
| 상태 양식 | `class AgentState(TypedDict): messages: Annotated[list[BaseMessage], add_messages]` |
| 노드 | `def node(state: AgentState) -> dict: return {'바꿀 키': 값}` |
| 설계도 | `StateGraph(AgentState)` · `add_node` · `add_edge(START, …)` · `add_edge(…, END)` |
| 실행 | `agent = graph.compile()` · `agent.invoke({...})` · `agent.stream({...}, stream_mode='updates')` |
| 메시지 짝 | `AIMessage.tool_calls[i]['id']` = `ToolMessage.tool_call_id` |